# XGBoost Super Dataset — Probability Threshold Tuning
**Author:** Akhila Annireddy  
**Purpose:** Apply per-drug probability threshold tuning on top of the already-trained XGBoost Super Dataset model.  
**Model used:** `xgboost_super_final_model.ubj` (demographics + prescription features)  
**No retraining:** This notebook only changes how predictions are interpreted — the saved model is loaded and used as-is.  

## Context

The XGBoost Super Dataset model already recalled **201 / 217 drugs** before any threshold tuning,  
with macro recall of 0.2925 on MEPS 2022. This is a massive improvement over the base XGBoost  
(demographics only) which recalled ~67 drugs before threshold tuning and 102 after.

## What Is Probability Threshold Tuning?

Normally XGBoost predicts whichever drug has the highest probability for each row.  
Example: if atorvastatin = 40% and ivermectin = 2%, it always predicts atorvastatin.  

With threshold tuning, we lower the bar for rare drugs.  
Example: if ivermectin's threshold is set to 1%, then its 2% probability is enough to predict it.  

**How thresholds are set:**  
Threshold for drug d = sqrt(freq(d) / max_freq)  
Common drugs keep threshold ≈ 1.0, rarest drugs get threshold as low as ~0.017.  
sqrt softens the ratio — same reasoning as sqrt sample weights in the weighted XGBoost.  

**What this trades off:**  
- Recall goes UP for rare drugs (model predicts them more often)  
- Precision goes DOWN (model may predict rare drugs incorrectly at times)  
- This is the right tradeoff — missing a drug = underestimating pharmaceutical pollution  

## Key difference from base threshold tuning notebook

The super dataset model was trained with prescription features (Quantity, Form, Strength, Day_Supply).  
The 2022 data used here must be `super_data_2022.csv` (not `data_2022.csv`) so it includes those same features.  
Preprocessing mirrors training: demographic NaNs filled with training medians, prescription NaNs left as NaN.

In [ ]:
!pip install 'xgboost>=2.1.0' permetrics

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, matthews_corrcoef,
    classification_report
)
from sklearn.utils import shuffle
from permetrics import ClassificationMetric
import warnings
warnings.filterwarnings('ignore')
import joblib

print("XGBoost version:", xgb.__version__)

XGBoost version: 2.1.4


In [ ]:
# Load the super integrated training data (2014-2021) — used only to:
#   1. Fit LabelEncoder with same drug->integer mapping as training
#   2. Compute drug frequency counts for threshold calculation
#   3. Get Family_income median for demographic imputation on 2022 data
#
# Load the super 2022 data — this is the held-out test set for threshold evaluation.
# Must be the super version (demographics + prescription features).

DATA_DIR = './'  # change this to your data path if needed

super_df = pd.read_csv(
    f'{DATA_DIR}super_integrated_data.csv',
    sep=None,
    engine='python',
    encoding='utf-8-sig'
)

if 'Unnamed: 0' in super_df.columns:
    super_df = super_df.drop(columns=['Unnamed: 0'])

data_2022 = pd.read_csv(
    f'{DATA_DIR}super_data_2022.csv',
    sep=None,
    engine='python',
    encoding='utf-8-sig'
)
if 'Unnamed: 0' in data_2022.columns:
    data_2022 = data_2022.drop(columns=['Unnamed: 0'])

print("Super training data shape:", super_df.shape)
print("Super 2022 data shape:", data_2022.shape)
print("\nTraining columns:", super_df.columns.tolist())
print("2022 columns:", data_2022.columns.tolist())

Super training data shape: (905728, 11)
Super 2022 data shape: (175669, 11)

Training columns: ['Observation_ID', 'Drug', 'Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity', 'Quantity', 'Form', 'Strength', 'Day_Supply']
2022 columns: ['Observation_ID', 'Drug', 'Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity', 'Quantity', 'Form', 'Strength', 'Day_Supply']


In [ ]:
# Imputation functions for Age, Strength, and Day_Supply.
# Applied within each fold to prevent data leakage.
# Consistent with TabICL and all other base models.
# Note: Quantity NaNs are NOT imputed — they are structurally meaningful
# and left as NaN for XGBoost native handling.

def impute_age(df):
    df = df.copy()
    df['income_bracket'] = (
        df.groupby('Year')['Family_income']
        .transform(lambda x: pd.qcut(x, 4, labels=False, duplicates='drop') + 1)
    )
    hh_meds    = df.groupby(['Year', 'Household_ID'])['Age'].transform('median')
    grp_meds   = df.groupby(['Year', 'income_bracket', 'Insurance_coverage'])['Age'].transform('median')
    yr_meds    = df.groupby('Year')['Age'].transform('median')
    global_med = df['Age'].median()
    df['Age']  = df['Age'].fillna(hh_meds)
    df['Age']  = df['Age'].fillna(grp_meds)
    df['Age']  = df['Age'].fillna(yr_meds)
    df['Age']  = df['Age'].fillna(global_med)
    df.drop(columns=['income_bracket'], inplace=True)
    return df

def impute_strength(df):
    df = df.copy()
    mask = df['Drug'] != 'no prescriptions'
    # 1. Median by Year and Drug
    yr_drug_meds = df.groupby(['Year', 'Drug'])['Strength'].transform('median')
    # 2. Median by Drug (across all years)
    drug_meds = df.groupby('Drug')['Strength'].transform('median')
    # 3. Global Median
    global_med   = df.loc[mask, 'Strength'].median()

    df.loc[mask, 'Strength'] = df.loc[mask, 'Strength'].fillna(yr_drug_meds)
    df.loc[mask, 'Strength'] = df.loc[mask, 'Strength'].fillna(drug_meds)
    df.loc[mask, 'Strength'] = df.loc[mask, 'Strength'].fillna(global_med)
    return df

def impute_day_supply(df):
    df = df.copy()
    mask = df['Drug'] != 'no prescriptions'
    # 1. Median by Year and Drug
    yr_drug_meds = df.groupby(['Year', 'Drug'])['Day_Supply'].transform('median')
    # 2. Median by Drug
    drug_meds = df.groupby('Drug')['Day_Supply'].transform('median')
    # 3. Global Median
    global_med   = df.loc[mask, 'Day_Supply'].median()

    df.loc[mask, 'Day_Supply'] = df.loc[mask, 'Day_Supply'].fillna(yr_drug_meds)
    df.loc[mask, 'Day_Supply'] = df.loc[mask, 'Day_Supply'].fillna(drug_meds)
    df.loc[mask, 'Day_Supply'] = df.loc[mask, 'Day_Supply'].fillna(global_med)
    return df

In [ ]:
# Define columns — must match exactly what the super dataset model was trained on.
# Quantity NaNs in 2022 data are left as NaN (same as training — XGBoost handles natively).
# Age, Strength, Day_Supply imputed using same functions as training.
# Family_income filled with training-set median.

feature_cols      = ['Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity',
                     'Quantity', 'Form', 'Strength', 'Day_Supply']
categorical_cols  = ['Sex', 'Insurance_coverage', 'Race_ethnicity', 'Form']
numeric_rx_cols       = ['Quantity', 'Strength', 'Day_Supply']
target_col        = 'Drug'

# set structural NaN's to -1 (this is necessary to differentiate between random
# missingness and the structural missingness of "no prescriptions")
mask = super_df['Drug'] == 'no prescriptions'
super_df.loc[mask, numeric_rx_cols] = -1
super_df.loc[mask, 'Form'] = '-1'

# Fit LabelEncoder on full training data — same mapping as xgboost_super_dataset.ipynb
le = LabelEncoder()
le.fit_transform(super_df[target_col])
print("Unique classes in encoder:", len(le.classes_))

for col in categorical_cols:
        super_df[col] = super_df[col].astype('category')

# impute for random missing values
super_df = impute_age(super_df)
super_df = impute_strength(super_df)
super_df = impute_day_supply(super_df)

# Filter 2022 to only known drugs
known_drugs        = set(le.classes_)
unseen_drugs       = set(data_2022[target_col].unique()) - known_drugs
data_2022_filtered = data_2022[data_2022[target_col].isin(known_drugs)].copy()
print(f"Unseen drugs in 2022 (dropped): {len(unseen_drugs)}")
print(f"2022 rows after filtering: {len(data_2022_filtered):,}")

# fill sentinel value for structural NaNs
mask = data_2022['Drug'] == 'no prescriptions'
data_2022.loc[mask, numeric_rx_cols] = data_2022.loc[mask, numeric_rx_cols].fillna(-1)
data_2022.loc[mask, 'Form'] = data_2022.loc[mask, 'Form'].fillna('-1')

# impute for random missing values
data_2022_filtered = impute_age(data_2022)
data_2022_filtered = impute_strength(data_2022)
data_2022_filtered = impute_day_supply(data_2022)

# Prepare 2022 features
X_2022 = data_2022_filtered[feature_cols].copy()
for col in categorical_cols:
    X_2022[col] = X_2022[col].astype('category')

y_2022_encoded = le.transform(data_2022_filtered[target_col])

# Shuffle to separate refill records.
X_2022, y_2022_encoded = shuffle(X_2022, y_2022_encoded, random_state=42)
X_2022 = X_2022.reset_index(drop=True)

# Quantity NaNs passed through as NaN — XGBoost learns missing direction natively.
d2022 = xgb.DMatrix(X_2022, label=y_2022_encoded, enable_categorical=True)
print("\n2022 DMatrix ready.")

Unique classes in encoder: 217
Training medians (Age, Family_income): {'Age': 60.0, 'Family_income': 38865.0}
Unseen drugs in 2022 (dropped): 0
2022 rows after filtering: 175,669

2022 DMatrix ready.


In [ ]:
# Compute per-drug probability thresholds based on training data frequency.
#
# Formula: threshold(d) = sqrt(freq(d) / max_freq)
#   - Most common drug (no prescriptions, 97497 rows) → threshold = sqrt(1.0) = 1.000
#   - Rarest drug (ivermectin, 29 rows) → threshold = sqrt(29/97497) = 0.017
#
# To apply: divide each drug's probability column by its threshold, then argmax.
# This boosts rare drug probabilities relative to common ones without retraining.
#
# Note: thresholds are computed from training data drug counts (2014-2021),
# not from the 2022 test set — no leakage.

drug_counts = super_df[target_col].value_counts()
max_freq    = drug_counts.max()

thresholds = np.array([
    np.sqrt(drug_counts.get(drug, 1) / max_freq)
    for drug in le.classes_
])

print("Threshold stats:")
print(f"  Max threshold (most common drug):  {thresholds.max():.4f}")
print(f"  Min threshold (rarest drug):       {thresholds.min():.4f}")
print(f"  Ratio max/min:                     {thresholds.max()/thresholds.min():.1f}x")

print("\nExample thresholds:")
for drug in ['no prescriptions', 'atorvastatin', 'lisinopril', 'metformin',
             'ivermectin', 'piroxicam', 'gentamicin']:
    if drug in le.classes_:
        idx = np.where(le.classes_ == drug)[0][0]
        print(f"  {drug:25s}: {thresholds[idx]:.4f}")

Threshold stats:
  Max threshold (most common drug):  1.0000
  Min threshold (rarest drug):       0.0172
  Ratio max/min:                     58.0x

Example thresholds:
  no prescriptions         : 1.0000
  atorvastatin             : 0.6289
  lisinopril               : 0.6064
  metformin                : 0.5886
  ivermectin               : 0.0172
  piroxicam                : 0.0250
  gentamicin               : 0.0256


In [ ]:
# Helper function — identical logic to base threshold tuning notebook.
# Reusable for any model that produces softprob output.

def apply_threshold_and_evaluate(model, dmatrix, y_true, thresholds, le, label):
    """
    Apply probability threshold tuning to a trained XGBoost model.

    Steps:
    1. Get softprob output — probabilities for all 218 classes per row
    2. Divide each drug's probability by its threshold
       (boosts rare drug probabilities relative to common ones)
    3. Take argmax of adjusted probabilities as final prediction
    4. Compute all required metrics and compare to hard (no-threshold) predictions
    """
    n_classes = len(le.classes_)

    probs = model.predict(dmatrix)

    # verify correct shape
    print(f"\n{'='*50}")
    print(f"{label}")
    print(f"{'='*50}")
    print(f"Probability matrix shape: {probs.shape}")

    # --- Hard predictions (no threshold) ---
    y_pred_hard = np.argmax(probs, axis=1)

    acc_hard          = accuracy_score(y_true, y_pred_hard)
    mcc_hard          = matthews_corrcoef(y_true, y_pred_hard)
    ev_hard           = ClassificationMetric(y_true, y_pred_hard)
    macro_recall_hard = ev_hard.recall_score(average='macro')
    micro_recall_hard = ev_hard.recall_score(average='micro')
    macro_f2_hard     = ev_hard.fbeta_score(beta=2, average='macro')

    print("\nHard predictions (no threshold):")
    print(f"  Accuracy:     {acc_hard:.4f}")
    print(f"  MCC:          {mcc_hard:.4f}")
    print(f"  Macro Recall: {macro_recall_hard:.4f}")
    print(f"  Micro Recall: {micro_recall_hard:.4f}")
    print(f"  Macro F2:     {macro_f2_hard:.4f}")

    # Count drugs with non-zero recall before threshold
    report_hard = classification_report(
        y_true, y_pred_hard,
        labels=np.arange(n_classes),
        target_names=le.classes_,
        output_dict=True,
        zero_division=0
    )
    drugs_recalled_hard = sum(
        1 for drug in le.classes_
        if drug in report_hard and report_hard[drug]['recall'] > 0
    )
    print(f"  Drugs with recall > 0: {drugs_recalled_hard} / {n_classes}")

    # --- Threshold-adjusted predictions ---
    # Dividing probability by threshold < 1 boosts that drug's adjusted score.
    # Rare drugs (low threshold) get boosted most — model predicts them more willingly.
    adjusted_probs = probs / thresholds[np.newaxis, :]  # broadcast across rows
    y_pred_thresh  = np.argmax(adjusted_probs, axis=1)

    acc_thresh          = accuracy_score(y_true, y_pred_thresh)
    mcc_thresh          = matthews_corrcoef(y_true, y_pred_thresh)
    ev_thresh           = ClassificationMetric(y_true, y_pred_thresh)
    macro_recall_thresh = ev_thresh.recall_score(average='macro')
    micro_recall_thresh = ev_thresh.recall_score(average='micro')
    macro_prec_thresh   = ev_thresh.precision_score(average='macro')
    macro_f2_thresh     = ev_thresh.fbeta_score(beta=2, average='macro')

    print("\nThreshold-adjusted predictions:")
    print(f"  Accuracy:        {acc_thresh:.4f}  (change: {acc_thresh-acc_hard:+.4f})")
    print(f"  MCC:             {mcc_thresh:.4f}  (change: {mcc_thresh-mcc_hard:+.4f})")
    print(f"  Macro Recall:    {macro_recall_thresh:.4f}  (change: {macro_recall_thresh-macro_recall_hard:+.4f})")
    print(f"  Micro Recall:    {micro_recall_thresh:.4f}  (change: {micro_recall_thresh-micro_recall_hard:+.4f})")
    print(f"  Macro Precision: {macro_prec_thresh:.4f}")
    print(f"  Macro F2:        {macro_f2_thresh:.4f}  (change: {macro_f2_thresh-macro_f2_hard:+.4f})")

    # Per-drug metrics after thresholding
    report_thresh = classification_report(
        y_true, y_pred_thresh,
        labels=np.arange(n_classes),
        target_names=le.classes_,
        output_dict=True,
        zero_division=0
    )
    drug_metrics = pd.DataFrame([
        {'Drug':             drug,
         'Recall_thresh':    report_thresh[drug]['recall'],
         'Precision_thresh': report_thresh[drug]['precision'],
         'F2_thresh':        report_thresh[drug]['f1-score'],
         'Support':          report_thresh[drug]['support']}
        for drug in le.classes_ if drug in report_thresh
    ]).sort_values('Recall_thresh', ascending=False)

    print("\nTop 15 drugs by recall after threshold tuning:")
    print(drug_metrics.head(15).to_string(index=False))

    print("\nBottom 15 drugs — still zero recall after threshold tuning:")
    zero_recall = drug_metrics[drug_metrics['Recall_thresh'] == 0]
    print(zero_recall.to_string(index=False))

    drugs_recalled = len(drug_metrics[drug_metrics['Recall_thresh'] > 0])
    print(f"\nDrugs with recall > 0 after thresholding: {drugs_recalled} / {n_classes}")
    print(f"Additional drugs recovered by threshold tuning: {drugs_recalled - drugs_recalled_hard}")

    return {
        'label':                  label,
        'accuracy_hard':          acc_hard,
        'macro_recall_hard':      macro_recall_hard,
        'macro_f2_hard':          macro_f2_hard,
        'drugs_recalled_hard':    drugs_recalled_hard,
        'accuracy_thresh':        acc_thresh,
        'mcc_thresh':             mcc_thresh,
        'macro_recall_thresh':    macro_recall_thresh,
        'micro_recall_thresh':    micro_recall_thresh,
        'macro_prec_thresh':      macro_prec_thresh,
        'macro_f2_thresh':        macro_f2_thresh,
        'drugs_recalled_thresh':  drugs_recalled,
        'drugs_recovered':        drugs_recalled - drugs_recalled_hard,
    }, drug_metrics

## XGBoost Super Dataset + Threshold Tuning

Load the super dataset final model and apply threshold tuning.
The model already recalled 201 / 218 classes before any threshold adjustment.
Goal: push remaining 17 classes toward non-zero recall and improve macro recall overall.

In [ ]:
# Load the super dataset XGBoost final model.
# Model was saved with multi:softmax (outputs winning class integer).
# Override to multi:softprob so predict() returns a full probability
# distribution across all 218 classes — shape (n_rows, 218).
# Tree structure is unchanged — only the output format changes.

super_model = xgb.Booster()
super_model.load_model('xgboost_super_final_model.ubj')
#super_model.set_param({'objective': 'multi:softprob', 'num_class': len(le.classes_)})
print("Super dataset model loaded with softprob output.")

super_results, super_drug_metrics = apply_threshold_and_evaluate(
    model      = super_model,
    dmatrix    = d2022,
    y_true     = y_2022_encoded,
    thresholds = thresholds,
    le         = le,
    label      = 'XGBoost SUPER DATASET — Threshold Tuning on MEPS 2022'
)

super_drug_metrics.to_csv('xgboost_super_threshold_per_drug.csv', index=False)
print("\nSuper dataset threshold per-drug metrics saved to xgboost_super_threshold_per_drug.csv")

Super dataset model loaded with softprob output.

XGBoost SUPER DATASET — Threshold Tuning on MEPS 2022
Probability matrix shape: (175669, 217)

Hard predictions (no threshold):
  Accuracy:     0.3544
  MCC:          0.3444
  Macro Recall: 0.2925
  Micro Recall: 0.3544
  Macro F2:     0.2754
  Drugs with recall > 0: 176 / 217

Threshold-adjusted predictions:
  Accuracy:        0.3054  (change: -0.0489)
  MCC:             0.3001  (change: -0.0444)
  Macro Recall:    0.3143  (change: +0.0218)
  Micro Recall:    0.3054  (change: -0.0489)
  Macro Precision: 0.2942
  Macro F2:        0.2773  (change: +0.0019)

Top 15 drugs by recall after threshold tuning:
            Drug  Recall_thresh  Precision_thresh  F2_thresh  Support
       lactulose       1.000000          0.841121   0.913706     90.0
     gemfibrozil       1.000000          0.376582   0.547126    119.0
      colchicine       1.000000          1.000000   1.000000    168.0
     clavulanate       1.000000          0.444444   0.615385

In [ ]:
# Full comparison table — all XGBoost versions across the project.
# Combines base XGBoost results (from xgboost_threshold_tuning.ipynb)
# with new super dataset results.
#
# Base XGBoost results are loaded from saved CSVs rather than re-running
# so this notebook is self-contained.

# Load base XGBoost comparison from prior notebook if available
try:
    base_comparison = pd.read_csv(
    'xgboost_all_versions_comparison.csv',
    sep=None,
    engine='python',
    encoding='utf-8-sig'
)
    print("Base XGBoost comparison loaded:")
    print(base_comparison.to_string(index=False))
    base_loaded = True
except FileNotFoundError:
    print("xgboost_all_versions_comparison.csv not found — showing super dataset results only.")
    base_loaded = False

# Build super dataset rows
super_rows = pd.DataFrame([
    {
        'Model':          'XGBoost Super (no threshold)',
        'Accuracy':        super_results['accuracy_hard'],
        'Macro_Recall':    super_results['macro_recall_hard'],
        'Macro_F2':        super_results['macro_f2_hard'],
        'Drugs_Recalled':  super_results['drugs_recalled_hard'],
    },
    {
        'Model':          'XGBoost Super + Threshold',
        'Accuracy':        super_results['accuracy_thresh'],
        'Macro_Recall':    super_results['macro_recall_thresh'],
        'Macro_F2':        super_results['macro_f2_thresh'],
        'Drugs_Recalled':  super_results['drugs_recalled_thresh'],
    },
])

if base_loaded:
    full_comparison = pd.concat([base_comparison, super_rows], ignore_index=True)
else:
    full_comparison = super_rows

print("\n" + "="*70)
print("FULL COMPARISON — All XGBoost Versions on MEPS 2022")
print("="*70)
print(full_comparison.to_string(index=False))

full_comparison.to_csv('xgboost_all_versions_comparison_super.csv', index=False)
print("\nFull comparison saved to xgboost_all_versions_comparison_super.csv")

xgboost_all_versions_comparison.csv not found — showing super dataset results only.

FULL COMPARISON — All XGBoost Versions on MEPS 2022
                       Model  Accuracy  Macro_Recall  Macro_F2  Drugs_Recalled
XGBoost Super (no threshold)  0.354360      0.292494  0.275401             176
   XGBoost Super + Threshold  0.305438      0.314268  0.277347             181

Full comparison saved to xgboost_all_versions_comparison_super.csv


In [ ]:
# Per-drug deep dive — compare recall before and after threshold tuning.
# Focus on the drugs that had zero recall before threshold tuning.

# Load pre-threshold per-drug metrics from the super dataset notebook
try:
    pre_threshold = pd.read_csv(
    "xgboost_super_2022_per_drug_metrics.csv",
    sep=None,
    engine='python',
    encoding='utf-8-sig'
)
    pre_threshold = pre_threshold.rename(columns={
        'Recall_2022': 'Recall_before_thresh',
        'Precision_2022': 'Precision_before_thresh'
    })

    # Merge with post-threshold results
    comparison_per_drug = pre_threshold[['Drug', 'Recall_before_thresh', 'Support_2022']].merge(
        super_drug_metrics[['Drug', 'Recall_thresh', 'Precision_thresh']],
        on='Drug', how='left'
    )
    comparison_per_drug['Recall_change'] = (
        comparison_per_drug['Recall_thresh'] - comparison_per_drug['Recall_before_thresh']
    ).round(4)

    # Drugs that had 0 recall before threshold — did tuning help?
    zero_before = comparison_per_drug[comparison_per_drug['Recall_before_thresh'] == 0].copy()
    zero_before = zero_before.sort_values('Recall_thresh', ascending=False)

    print("Drugs with zero recall BEFORE threshold tuning:")
    print(f"Total: {len(zero_before)}")
    print(zero_before.to_string(index=False))

    # Drugs that improved most from threshold tuning
    improved = comparison_per_drug[comparison_per_drug['Recall_change'] > 0].sort_values(
        'Recall_change', ascending=False
    )
    print(f"\nDrugs that improved from threshold tuning: {len(improved)}")
    print(improved.head(20).to_string(index=False))

    # Drugs that got worse (threshold caused over-prediction of them less)
    worsened = comparison_per_drug[comparison_per_drug['Recall_change'] < 0].sort_values(
        'Recall_change'
    )
    print(f"\nDrugs that worsened from threshold tuning: {len(worsened)}")
    print(worsened.head(20).to_string(index=False))

    comparison_per_drug.to_csv('xgboost_super_threshold_comparison_per_drug.csv', index=False)
    print("\nPer-drug before/after comparison saved.")

except FileNotFoundError:
    print("xgboost_super_2022_per_drug_metrics.csv not found.")
    print("Run xgboost_super_dataset.ipynb first to generate it.")

Drugs with zero recall BEFORE threshold tuning:
Total: 41
                  Drug  Recall_before_thresh  Support_2022  Recall_thresh  Precision_thresh  Recall_change
          trimethoprim                   0.0           4.0       0.750000          0.750000         0.7500
           solifenacin                   0.0         130.0       0.223077          0.024431         0.2231
        desvenlafaxine                   0.0         145.0       0.082759          0.021779         0.0828
            benazepril                   0.0         297.0       0.057239          0.008718         0.0572
    dexmethylphenidate                   0.0         363.0       0.052342          0.155738         0.0523
        cyproheptadine                   0.0          31.0       0.032258          0.029412         0.0323
            olmesartan                   0.0         484.0       0.016529          0.004437         0.0165
           minocycline                   0.0          74.0       0.013514          0.0

In [ ]:
# Save a clean summary of the super threshold results for the paper.

summary = pd.DataFrame([{
    'model':                     'XGBoost_Super_Threshold',
    'dataset':                   'MEPS_2022_internal_validation',
    'accuracy_hard':             super_results['accuracy_hard'],
    'macro_recall_hard':         super_results['macro_recall_hard'],
    'macro_f2_hard':             super_results['macro_f2_hard'],
    'drugs_recalled_hard':       super_results['drugs_recalled_hard'],
    'accuracy_thresh':           super_results['accuracy_thresh'],
    'macro_recall_thresh':       super_results['macro_recall_thresh'],
    'macro_precision_thresh':    super_results['macro_prec_thresh'],
    'macro_f2_thresh':           super_results['macro_f2_thresh'],
    'drugs_recalled_thresh':     super_results['drugs_recalled_thresh'],
    'drugs_recovered_by_thresh': super_results['drugs_recovered'],
}])

summary.to_csv('xgboost_super_threshold_summary.csv', index=False)
print("Summary saved to xgboost_super_threshold_summary.csv")
print("\nFinal summary:")
print(summary.T.to_string())

Summary saved to xgboost_super_threshold_summary.csv

Final summary:
                                                       0
model                            XGBoost_Super_Threshold
dataset                    MEPS_2022_internal_validation
accuracy_hard                                    0.35436
macro_recall_hard                               0.292494
macro_f2_hard                                   0.275401
drugs_recalled_hard                                  176
accuracy_thresh                                 0.305438
macro_recall_thresh                             0.314268
macro_precision_thresh                           0.29416
macro_f2_thresh                                 0.277347
drugs_recalled_thresh                                181
drugs_recovered_by_thresh                              5


In [ ]:
# save threshold mapping for use in inference scripts
threshold_map = dict(zip(le.classes_, thresholds))
joblib.dump(threshold_map, "xgboost_threshold_map.joblib")

## Summary of Outputs

| File | Contents |
|------|----------|
| `xgboost_super_threshold_per_drug.csv` | Per-drug recall/precision/F2 after threshold tuning on super model |
| `xgboost_super_threshold_comparison_per_drug.csv` | Before vs after threshold comparison per drug |
| `xgboost_super_threshold_summary.csv` | Overall summary metrics — hard vs threshold adjusted |
| `xgboost_all_versions_comparison_super.csv` | Full comparison across all XGBoost versions |

## Key Metrics to Watch

**`Drugs_Recalled`** — how many of 218 classes have recall > 0 after threshold tuning.
Super dataset model already had 201 before threshold tuning.
If threshold tuning recovers some of the remaining 17, that's the final coverage number for the paper.

**`Macro_Recall`** — macro recall on 2022 was 0.2925 before threshold tuning.
Threshold tuning trades some accuracy for higher macro recall — expected direction.

## Project Comparison Reference

| Model | Drugs Recalled | Macro Recall |
|---|---|---|
| Base XGBoost (demographics only) | ~67 | 0.0101 |
| Base XGBoost + Threshold | 97 | 0.0188 |
| Base XGBoost Weighted + Threshold | 102 | 0.0182 |
| **XGBoost Super (no threshold)** | **201** | **0.2925** |
| **XGBoost Super + Threshold** | **TBD** | **TBD** |

**Next step:** Pass `xgboost_super_proba_2022.csv` (generated in `xgboost_super_dataset.ipynb`) to the ensemble model (`ensemble_pharmshed.ipynb`) along with probability outputs from all other base models for soft voting ensemble construction.